# 🔠 HCR System — EMNIST Model Retraining (v2)

This notebook retrains the handwritten character recognition CNN with:

| Improvement | Old (v1) | New (v2) |
|---|---|---|
| **Dataset** | EMNIST Balanced (131K samples) | EMNIST ByMerge (814K samples) |
| **Architecture** | Plain CNN (32→64→128) | Residual CNN with SE blocks |
| **Augmentation** | RandomAffine only | RandomAffine + Cutout + Mixup |
| **Scheduler** | CosineAnnealing | OneCycleLR |
| **Regularization** | Dropout only | Dropout + Label Smoothing |
| **Epochs** | 20 | 50 |

### Why EMNIST ByMerge over Balanced?
Both have the **same 47 classes** (digits + uppercase + 11 lowercase that look different from uppercase). But:
- **Balanced**: 131,600 samples (artificially limited to equal-per-class)
- **ByMerge**: 814,255 samples (6× more data, real-world distribution)

More data → better generalization. We handle the class imbalance with weighted sampling.

---
**⚠️ Set Runtime → GPU (T4 or better) before running!**

## 1. Setup & Installation

In [ ]:
!pip install -q torchvision matplotlib seaborn scikit-learn tqdm

import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm
from collections import Counter

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Configuration

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────
CONFIG = {
    'batch_size': 256,
    'epochs': 50,
    'learning_rate': 3e-3,
    'weight_decay': 1e-4,
    'val_fraction': 0.1,
    'label_smoothing': 0.1,
    'mixup_alpha': 0.2,
    'cutout_size': 8,
    'num_workers': 2,
    'data_dir': './data',
    'checkpoint_dir': './checkpoints',
}

os.makedirs(CONFIG['data_dir'], exist_ok=True)
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

# ── EMNIST 47-class mapping (same as your backend) ────────────────────
EMNIST_BALANCED_MAPPING = [
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
    'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
    'a', 'b', 'd', 'e', 'f', 'g', 'h', 'n', 'q', 'r', 't',
]
NUM_CLASSES = len(EMNIST_BALANCED_MAPPING)  # 47

print(f'Number of classes: {NUM_CLASSES}')
print(f'Classes: {EMNIST_BALANCED_MAPPING}')

## 3. Dataset Loading & Validation

We load **EMNIST ByMerge** (814K samples, same 47 classes as Balanced) and validate it thoroughly.

In [ ]:
# ── EMNIST orientation fix ───────────────────────────────────────────────
# EMNIST images are stored transposed+flipped vs what humans expect.
def fix_emnist_orientation(img):
    """Fix the known EMNIST orientation quirk."""
    img = TF.rotate(img, -90)
    img = TF.hflip(img)
    return img


# ── Cutout augmentation ──────────────────────────────────────────────────
class Cutout:
    """Randomly mask out a square region of the image."""
    def __init__(self, size=8):
        self.size = size

    def __call__(self, img):
        h, w = img.shape[-2:]
        y = random.randint(0, h - 1)
        x = random.randint(0, w - 1)
        y1 = max(0, y - self.size // 2)
        y2 = min(h, y + self.size // 2)
        x1 = max(0, x - self.size // 2)
        x2 = min(w, x + self.size // 2)
        img[..., y1:y2, x1:x2] = 0
        return img


# ── Load raw dataset for validation first ───────────────────────────────
print('Downloading EMNIST ByMerge (this may take a few minutes)...')
raw_train = datasets.EMNIST(
    root=CONFIG['data_dir'], split='bymerge', train=True, download=True
)
raw_test = datasets.EMNIST(
    root=CONFIG['data_dir'], split='bymerge', train=False, download=True
)
print(f'\n✅ Train samples: {len(raw_train):,}')
print(f'✅ Test samples:  {len(raw_test):,}')
print(f'✅ Image shape:   {raw_train[0][0].size}')
print(f'✅ Label range:   {min(raw_train.targets.numpy())} – {max(raw_train.targets.numpy())}')
print(f'✅ Unique labels: {len(set(raw_train.targets.numpy()))}')

### 3a. Validate label distribution (check for class imbalance)

In [ ]:
# ── Validate class distribution ──────────────────────────────────────────
train_labels = raw_train.targets.numpy()
label_counts = Counter(train_labels)

fig, ax = plt.subplots(figsize=(16, 5))
classes = sorted(label_counts.keys())
counts = [label_counts[c] for c in classes]
names = [EMNIST_BALANCED_MAPPING[c] for c in classes]

bars = ax.bar(range(len(classes)), counts, color='steelblue', edgecolor='white')
ax.set_xticks(range(len(classes)))
ax.set_xticklabels(names, fontsize=9)
ax.set_xlabel('Class')
ax.set_ylabel('Number of Samples')
ax.set_title('EMNIST ByMerge — Training Set Class Distribution')

# Highlight the most and least populated classes
max_idx = counts.index(max(counts))
min_idx = counts.index(min(counts))
bars[max_idx].set_color('green')
bars[min_idx].set_color('red')

ax.legend([bars[max_idx], bars[min_idx]],
          [f'Max: {names[max_idx]} ({max(counts):,})',
           f'Min: {names[min_idx]} ({min(counts):,})'])

plt.tight_layout()
plt.show()

print(f'\nImbalance ratio (max/min): {max(counts)/min(counts):.1f}x')
print(f'Total training samples: {sum(counts):,}')

### 3b. Visual sanity check — verify orientation fix and labels are correct

In [ ]:
# ── Visualize samples to verify orientation and labeling ─────────────────
fig, axes = plt.subplots(4, 12, figsize=(18, 7))
fig.suptitle('Random Samples (orientation-corrected)', fontsize=14, fontweight='bold')

indices = random.sample(range(len(raw_train)), 48)
for i, idx in enumerate(indices):
    img, label = raw_train[idx]
    # Apply orientation fix
    img = fix_emnist_orientation(img)
    ax = axes[i // 12][i % 12]
    ax.imshow(np.array(img), cmap='gray')
    char = EMNIST_BALANCED_MAPPING[label]
    ax.set_title(f'{char} ({label})', fontsize=9, color='green')
    ax.axis('off')

plt.tight_layout()
plt.show()
print('\n✅ Check above: Do the images match their labels?')
print('   If letters appear rotated or flipped, the orientation fix is wrong.')

### 3c. Verify all 47 classes have data and compute normalization stats

In [ ]:
# ── Verify completeness and compute normalization constants ─────────────
assert len(set(train_labels)) == NUM_CLASSES, \
    f'Expected {NUM_CLASSES} classes, found {len(set(train_labels))}'

# Compute actual mean/std on the full ByMerge training set
print('Computing dataset mean and std (this takes ~30 seconds)...')
temp_transform = transforms.Compose([
    transforms.Lambda(fix_emnist_orientation),
    transforms.ToTensor(),
])
temp_ds = datasets.EMNIST(
    root=CONFIG['data_dir'], split='bymerge', train=True, download=False,
    transform=temp_transform
)
temp_loader = DataLoader(temp_ds, batch_size=1024, num_workers=2, shuffle=False)

pixel_sum, pixel_sq_sum, n_pixels = 0.0, 0.0, 0
for images, _ in tqdm(temp_loader, desc='Computing stats'):
    pixel_sum += images.sum().item()
    pixel_sq_sum += (images ** 2).sum().item()
    n_pixels += images.numel()

DATASET_MEAN = pixel_sum / n_pixels
DATASET_STD = (pixel_sq_sum / n_pixels - DATASET_MEAN ** 2) ** 0.5

print(f'\n✅ Dataset Mean: {DATASET_MEAN:.4f}')
print(f'✅ Dataset Std:  {DATASET_STD:.4f}')
print(f'✅ All {NUM_CLASSES} classes present in training data')

del temp_ds, temp_loader  # free memory

## 4. Build Data Loaders with Advanced Augmentation

In [ ]:
# ── Transforms ───────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Lambda(fix_emnist_orientation),
    transforms.RandomAffine(
        degrees=10,
        translate=(0.1, 0.1),
        scale=(0.85, 1.15),
        shear=8,
    ),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize((DATASET_MEAN,), (DATASET_STD,)),
    Cutout(size=CONFIG['cutout_size']),
])

eval_transform = transforms.Compose([
    transforms.Lambda(fix_emnist_orientation),
    transforms.ToTensor(),
    transforms.Normalize((DATASET_MEAN,), (DATASET_STD,)),
])


# ── Load datasets ─────────────────────────────────────────────────────────
train_full = datasets.EMNIST(
    root=CONFIG['data_dir'], split='bymerge', train=True,
    download=False, transform=train_transform
)
test_set = datasets.EMNIST(
    root=CONFIG['data_dir'], split='bymerge', train=False,
    download=False, transform=eval_transform
)

# Split training into train + validation
n_val = int(len(train_full) * CONFIG['val_fraction'])
n_train = len(train_full) - n_val
train_set, val_set = random_split(
    train_full, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

# Validation set should NOT use training augmentation
val_full = datasets.EMNIST(
    root=CONFIG['data_dir'], split='bymerge', train=True,
    download=False, transform=eval_transform
)
val_set_clean = torch.utils.data.Subset(val_full, val_set.indices)


# ── Weighted sampler to handle class imbalance ────────────────────────────
train_targets = [train_full.targets[i] for i in train_set.indices]
class_counts = Counter(t.item() if isinstance(t, torch.Tensor) else t for t in train_targets)
total = sum(class_counts.values())
class_weights = {c: total / (NUM_CLASSES * count) for c, count in class_counts.items()}
sample_weights = [class_weights[t.item() if isinstance(t, torch.Tensor) else t] for t in train_targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)


# ── DataLoaders ──────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_set, batch_size=CONFIG['batch_size'], sampler=sampler,
    num_workers=CONFIG['num_workers'], pin_memory=True
)
val_loader = DataLoader(
    val_set_clean, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=CONFIG['num_workers'], pin_memory=True
)
test_loader = DataLoader(
    test_set, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=CONFIG['num_workers'], pin_memory=True
)

print(f'Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_set):,}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

### 4a. Preview augmented samples

In [ ]:
# ── Preview what augmented training data looks like ────────────────────────
batch_imgs, batch_labels = next(iter(train_loader))

fig, axes = plt.subplots(3, 10, figsize=(16, 5))
fig.suptitle('Augmented Training Samples (with Cutout, Affine, Perspective)', fontweight='bold')

for i in range(30):
    ax = axes[i // 10][i % 10]
    img = batch_imgs[i].squeeze().numpy()
    # Unnormalize for visualization
    img = img * DATASET_STD + DATASET_MEAN
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(EMNIST_BALANCED_MAPPING[batch_labels[i]], fontsize=10, color='blue')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Model Architecture — Residual CNN with Squeeze-and-Excitation

This is a significant upgrade from the v1 plain CNN. Key improvements:
- **Residual connections** → trains deeper networks without vanishing gradients
- **SE (Squeeze-and-Excitation) blocks** → adaptively re-weights channel features
- **Larger capacity** → more filters (64→128→256) to learn complex character patterns

The output checkpoint is **fully compatible** with your backend — we save it in the same format.

In [ ]:
# ── Squeeze-and-Excitation Block ─────────────────────────────────────────
class SEBlock(nn.Module):
    """Channel attention: learns which feature channels are important."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        scale = self.squeeze(x).view(b, c)
        scale = self.excitation(scale).view(b, c, 1, 1)
        return x * scale


# ── Residual Block ───────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.se = SEBlock(out_ch)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += self.shortcut(x)
        return F.relu(out, inplace=True)


# ── Full Model ──────────────────────────────────────────────────────────
class EMNISTCNN_v2(nn.Module):
    """Residual CNN with SE blocks for EMNIST character recognition."""
    def __init__(self, num_classes=47):
        super().__init__()
        # Initial convolution: 1 → 64 channels
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        # Residual stages
        self.stage1 = nn.Sequential(
            ResidualBlock(64, 64),
            ResidualBlock(64, 64),
        )  # 28x28
        self.stage2 = nn.Sequential(
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128),
        )  # 14x14
        self.stage3 = nn.Sequential(
            ResidualBlock(128, 256, stride=2),
            ResidualBlock(256, 256),
        )  # 7x7
        # Head
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)


# ── Instantiate and inspect ──────────────────────────────────────────────
model = EMNISTCNN_v2(NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: EMNISTCNN_v2')
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Quick forward pass test
test_input = torch.randn(2, 1, 28, 28).to(device)
test_output = model(test_input)
print(f'Input shape:  {test_input.shape}')
print(f'Output shape: {test_output.shape}')
assert test_output.shape == (2, NUM_CLASSES), 'Output shape mismatch!'
print('✅ Model architecture verified')

## 6. Training Loop with Mixup

In [ ]:
# ── Mixup helper ─────────────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.2):
    """Create mixed inputs and targets for Mixup augmentation."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Compute mixed loss for Mixup."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ── Training function ────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scheduler, use_mixup=True):
    model.train()
    total_loss, total_correct, total_n = 0.0, 0, 0

    pbar = tqdm(loader, desc='  Train', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        if use_mixup and CONFIG['mixup_alpha'] > 0:
            mixed_x, y_a, y_b, lam = mixup_data(images, labels, CONFIG['mixup_alpha'])
            logits = model(mixed_x)
            loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
            # For accuracy tracking, use the dominant label
            _, predicted = logits.max(1)
            correct = (lam * predicted.eq(y_a).float() +
                       (1 - lam) * predicted.eq(y_b).float()).sum().item()
        else:
            logits = model(images)
            loss = criterion(logits, labels)
            correct = (logits.argmax(1) == labels).sum().item()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * images.size(0)
        total_correct += correct
        total_n += images.size(0)

        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{total_correct/total_n:.4f}')

    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_n += images.size(0)

    return total_loss / total_n, total_correct / total_n

## 7. Train the Model

In [ ]:
# ── Setup optimizer, scheduler, loss ──────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=CONFIG['learning_rate'],
    steps_per_epoch=len(train_loader),
    epochs=CONFIG['epochs'],
    pct_start=0.1,
    anneal_strategy='cos',
)

# ── Training history ──────────────────────────────────────────────────────
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'lr': [],
}

best_val_acc = 0.0
best_epoch = 0
patience = 10
no_improve = 0

print(f'Training for {CONFIG["epochs"]} epochs...')
print(f'Using: Mixup(α={CONFIG["mixup_alpha"]}), Cutout({CONFIG["cutout_size"]}px), '
      f'LabelSmoothing({CONFIG["label_smoothing"]})')
print('=' * 80)

start_time = time.time()

for epoch in range(1, CONFIG['epochs'] + 1):
    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, scheduler, use_mixup=True
    )
    # Validate
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    # Record
    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    # Save best
    improved = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        no_improve = 0
        improved = ' ⭐ NEW BEST'
        torch.save({
            'model_state': model.state_dict(),
            'num_classes': NUM_CLASSES,
            'mapping': EMNIST_BALANCED_MAPPING,
            'norm_mean': DATASET_MEAN,
            'norm_std': DATASET_STD,
            'val_acc': val_acc,
            'epoch': epoch,
            'architecture': 'EMNISTCNN_v2',
            'dataset': 'EMNIST_ByMerge',
            'config': CONFIG,
        }, os.path.join(CONFIG['checkpoint_dir'], 'emnist_balanced.pt'))
    else:
        no_improve += 1

    elapsed = time.time() - start_time
    print(f'Epoch {epoch:3d}/{CONFIG["epochs"]} │ '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} │ '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} │ '
          f'LR: {current_lr:.6f} │ '
          f'Time: {elapsed/60:.1f}m{improved}')

    # Early stopping
    if no_improve >= patience:
        print(f'\n⏹ Early stopping at epoch {epoch} (no improvement for {patience} epochs)')
        break

total_time = time.time() - start_time
print(f'\n{"=" * 80}')
print(f'Training complete in {total_time/60:.1f} minutes')
print(f'Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}')

## 8. Training Curves

In [ ]:
# ── Plot training history ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', color='#2196F3', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', color='#FF5722', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot([a * 100 for a in history['train_acc']], label='Train', color='#2196F3', linewidth=2)
axes[1].plot([a * 100 for a in history['val_acc']], label='Validation', color='#FF5722', linewidth=2)
axes[1].axhline(y=best_val_acc * 100, color='green', linestyle='--', alpha=0.5,
                label=f'Best: {best_val_acc*100:.2f}%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning Rate
axes[2].plot(history['lr'], color='#9C27B0', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule (OneCycleLR)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Evaluate on Test Set

In [ ]:
# ── Load the best checkpoint ─────────────────────────────────────────────
best_ckpt = torch.load(
    os.path.join(CONFIG['checkpoint_dir'], 'emnist_balanced.pt'),
    map_location=device, weights_only=False
)
model.load_state_dict(best_ckpt['model_state'])
print(f'Loaded best checkpoint from epoch {best_ckpt["epoch"]} '
      f'(val_acc={best_ckpt["val_acc"]:.4f})')

# ── Final test evaluation ────────────────────────────────────────────────
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f'\n🎯 Test Loss:     {test_loss:.4f}')
print(f'🎯 Test Accuracy: {test_acc*100:.2f}%')

### 9a. Per-class accuracy and confusion matrix

In [ ]:
# ── Collect predictions for detailed analysis ─────────────────────────────
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Test evaluation'):
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

# Classification report
print('\nPer-Class Classification Report:')
print('=' * 60)
report = classification_report(
    all_labels, all_preds,
    target_names=EMNIST_BALANCED_MAPPING,
    digits=4
)
print(report)

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(
    cm_normalized, annot=False, fmt='.2f', cmap='Blues',
    xticklabels=EMNIST_BALANCED_MAPPING,
    yticklabels=EMNIST_BALANCED_MAPPING,
    ax=ax, vmin=0, vmax=1
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Normalized Confusion Matrix (Test Acc: {test_acc*100:.2f}%)', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Find the most confused pairs ──────────────────────────────────────────
print('\n🔍 Most Confused Character Pairs (off-diagonal max):')
np.fill_diagonal(cm_normalized, 0)
for _ in range(10):
    i, j = np.unravel_index(cm_normalized.argmax(), cm_normalized.shape)
    print(f'  {EMNIST_BALANCED_MAPPING[i]} → {EMNIST_BALANCED_MAPPING[j]}: '
          f'{cm_normalized[i, j]*100:.1f}% misclassified')
    cm_normalized[i, j] = 0

## 10. Export — Backend-Compatible Checkpoint

**IMPORTANT**: The v2 model has a different architecture than v1, so we need to also export the model class.

We save two files:
1. `emnist_balanced.pt` — the checkpoint (weights + metadata), same format as your backend expects
2. `emnist_model_v2.py` — the updated model architecture to replace your backend's `emnist_model.py`

In [ ]:
# ── Save final checkpoint ────────────────────────────────────────────────
final_path = os.path.join(CONFIG['checkpoint_dir'], 'emnist_balanced.pt')
print(f'Checkpoint saved: {final_path}')
print(f'  Architecture: EMNISTCNN_v2 (Residual + SE)')
print(f'  Dataset:      EMNIST ByMerge ({len(raw_train):,} train samples)')
print(f'  Classes:      {NUM_CLASSES}')
print(f'  Val Accuracy: {best_val_acc*100:.2f}%')
print(f'  Test Accuracy: {test_acc*100:.2f}%')
print(f'  Mean/Std:     {DATASET_MEAN:.4f} / {DATASET_STD:.4f}')

# ── Also save the model architecture as a standalone Python file ────────
model_code = '''"""\nEMNIST CNN v2 — Residual CNN with Squeeze-and-Excitation blocks.\n\nDrop-in replacement for the original emnist_model.py in backend/ml/.\nInput:  (N, 1, 28, 28) grayscale images, normalized.\nOutput: (N, 47) logits, indexed per EMNIST_BALANCED_MAPPING.\n"""\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n# ── EMNIST Balanced label mapping (47 classes) ──────────────────────────\nEMNIST_BALANCED_MAPPING = [\n    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',\n    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',\n    'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',\n    'a', 'b', 'd', 'e', 'f', 'g', 'h', 'n', 'q', 'r', 't',\n]\n\nNUM_CLASSES = len(EMNIST_BALANCED_MAPPING)  # 47\n\n# Convenience subsets\nDIGIT_INDICES = list(range(0, 10))\nUPPERCASE_INDICES = list(range(10, 36))\nLOWERCASE_INDICES = list(range(36, 47))\n\n\ndef index_to_char(idx: int) -> str:\n    return EMNIST_BALANCED_MAPPING[idx]\n\n\nclass SEBlock(nn.Module):\n    """Channel attention: learns which feature channels are important."""\n    def __init__(self, channels, reduction=4):\n        super().__init__()\n        self.squeeze = nn.AdaptiveAvgPool2d(1)\n        self.excitation = nn.Sequential(\n            nn.Linear(channels, channels // reduction, bias=False),\n            nn.ReLU(inplace=True),\n            nn.Linear(channels // reduction, channels, bias=False),\n            nn.Sigmoid(),\n        )\n\n    def forward(self, x):\n        b, c, _, _ = x.shape\n        scale = self.squeeze(x).view(b, c)\n        scale = self.excitation(scale).view(b, c, 1, 1)\n        return x * scale\n\n\nclass ResidualBlock(nn.Module):\n    def __init__(self, in_ch, out_ch, stride=1):\n        super().__init__()\n        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)\n        self.bn1 = nn.BatchNorm2d(out_ch)\n        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)\n        self.bn2 = nn.BatchNorm2d(out_ch)\n        self.se = SEBlock(out_ch)\n\n        self.shortcut = nn.Sequential()\n        if stride != 1 or in_ch != out_ch:\n            self.shortcut = nn.Sequential(\n                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),\n                nn.BatchNorm2d(out_ch),\n            )\n\n    def forward(self, x):\n        out = F.relu(self.bn1(self.conv1(x)), inplace=True)\n        out = self.bn2(self.conv2(out))\n        out = self.se(out)\n        out += self.shortcut(x)\n        return F.relu(out, inplace=True)\n\n\nclass EMNISTCNN(nn.Module):\n    """Residual CNN with SE blocks for EMNIST character recognition.\n    \n    Drop-in compatible: same class name (EMNISTCNN) as v1, same\n    checkpoint format, same input/output shapes.\n    """\n    def __init__(self, num_classes: int = NUM_CLASSES):\n        super().__init__()\n        self.stem = nn.Sequential(\n            nn.Conv2d(1, 64, 3, padding=1, bias=False),\n            nn.BatchNorm2d(64),\n            nn.ReLU(inplace=True),\n        )\n        self.stage1 = nn.Sequential(\n            ResidualBlock(64, 64),\n            ResidualBlock(64, 64),\n        )\n        self.stage2 = nn.Sequential(\n            ResidualBlock(64, 128, stride=2),\n            ResidualBlock(128, 128),\n        )\n        self.stage3 = nn.Sequential(\n            ResidualBlock(128, 256, stride=2),\n            ResidualBlock(256, 256),\n        )\n        self.pool = nn.AdaptiveAvgPool2d(1)\n        self.classifier = nn.Sequential(\n            nn.Dropout(0.3),\n            nn.Linear(256, num_classes),\n        )\n\n    # Keep old attribute names for backward compat with recognition_engine\n    @property\n    def features(self):\n        return nn.Sequential(self.stem, self.stage1, self.stage2, self.stage3)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.stem(x)\n        x = self.stage1(x)\n        x = self.stage2(x)\n        x = self.stage3(x)\n        x = self.pool(x).flatten(1)\n        return self.classifier(x)\n'''

model_path = os.path.join(CONFIG['checkpoint_dir'], 'emnist_model.py')
with open(model_path, 'w') as f:
    f.write(model_code)
print(f'\nModel architecture saved: {model_path}')

print('\n' + '=' * 60)
print('📋 TO DEPLOY:')
print('  1. Download checkpoints/emnist_balanced.pt')
print('  2. Download checkpoints/emnist_model.py')
print('  3. Replace backend/ml/emnist_model.py with the new one')
print('  4. Copy emnist_balanced.pt into backend/ml/checkpoints/')
print('  5. Push to GitHub → HuggingFace Space auto-rebuilds')
print('=' * 60)

## 11. Test on Sample Images

Quick visual test to verify the model works correctly on individual characters.

In [ ]:
# ── Visual predictions on test set ────────────────────────────────────────
model.eval()
test_imgs, test_labels = next(iter(test_loader))

fig, axes = plt.subplots(4, 10, figsize=(18, 8))
fig.suptitle('Predictions on Test Samples', fontsize=14, fontweight='bold')

with torch.no_grad():
    logits = model(test_imgs[:40].to(device))
    probs = torch.softmax(logits, dim=1)
    confs, preds = probs.max(dim=1)

for i in range(40):
    ax = axes[i // 10][i % 10]
    img = test_imgs[i].squeeze().numpy()
    img = img * DATASET_STD + DATASET_MEAN
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)

    pred_char = EMNIST_BALANCED_MAPPING[preds[i].item()]
    true_char = EMNIST_BALANCED_MAPPING[test_labels[i].item()]
    conf = confs[i].item()

    color = 'green' if pred_char == true_char else 'red'
    ax.set_title(f'{pred_char} ({conf:.0%})', fontsize=9, color=color)
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Green = correct, Red = incorrect')

## 12. Download Files

Run the cell below to download the trained checkpoint and model file from Colab.

In [ ]:
# ── Download to your local machine ────────────────────────────────────────
try:
    from google.colab import files
    print('Downloading checkpoint...')
    files.download(os.path.join(CONFIG['checkpoint_dir'], 'emnist_balanced.pt'))
    print('Downloading model architecture...')
    files.download(os.path.join(CONFIG['checkpoint_dir'], 'emnist_model.py'))
    print('\n✅ Both files downloaded!')
except ImportError:
    print('Not running on Colab. Files are in:', CONFIG['checkpoint_dir'])
    print('Copy them manually to your backend/ml/ directory.')